In [1]:
!apt-get update -qq
!apt-get install -y ns2

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following additional packages will be installed:
  libotcl1 libpcap0.8t64 libtclcl1
Suggested packages:
  gnuplot
The following NEW packages will be installed:
  libotcl1 libpcap0.8t64 libtclcl1 ns2
0 upgraded, 4 newly installed, 0 to remove and 68 not upgraded.
Need to get 2,810 kB of archives.
After this operation, 15.7 MB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu noble-updates/main amd64 libpcap0.8t64 amd64 1.10.4-4.1ubuntu3.1 [151 kB]
Get:2 http://archive.ubuntu.com/ubuntu noble/universe amd64 libotcl1 amd64 1.14+dfsg-8 [24.2 kB]
Get:3 http://archive.ubuntu.com/ubuntu noble/universe amd64 libtclcl1 amd64 1.20-10 [65.2 kB]
Get:4 http://archive.ubuntu.com/ubuntu n

In [2]:
%%writefile exp1.tcl

set ns [new Simulator]

set tr [open out.tr w]
set nf [open out.nam w]

$ns trace-all $tr
$ns namtrace-all $nf

set n0 [$ns node]
set n1 [$ns node]

$ns duplex-link $n0 $n1 1Mb 10ms DropTail

set udp [new Agent/UDP]
set null [new Agent/Null]

$ns attach-agent $n0 $udp
$ns attach-agent $n1 $null
$ns connect $udp $null

set cbr [new Application/Traffic/CBR]
$cbr attach-agent $udp

$cbr set packetSize_ 512
$cbr set interval_ 0.05

$ns at 0.5 "$cbr start"
$ns at 4.5 "$cbr stop"

$ns at 5.0 "finish"

proc finish {} {
    global ns tr nf
    $ns flush-trace
    close $tr
    close $nf
    exit 0
}

$ns run

Writing exp1.tcl


In [3]:
!ns exp1.tcl

In [4]:
!head out.tr

+ 0.5 0 1 cbr 512 ------- 0 0.0 1.0 0 0
- 0.5 0 1 cbr 512 ------- 0 0.0 1.0 0 0
r 0.514096 0 1 cbr 512 ------- 0 0.0 1.0 0 0
+ 0.55 0 1 cbr 512 ------- 0 0.0 1.0 1 1
- 0.55 0 1 cbr 512 ------- 0 0.0 1.0 1 1
r 0.564096 0 1 cbr 512 ------- 0 0.0 1.0 1 1
+ 0.6 0 1 cbr 512 ------- 0 0.0 1.0 2 2
- 0.6 0 1 cbr 512 ------- 0 0.0 1.0 2 2
r 0.614096 0 1 cbr 512 ------- 0 0.0 1.0 2 2
+ 0.65 0 1 cbr 512 ------- 0 0.0 1.0 3 3


In [5]:
!awk '$1=="d" {n++} END {print "Dropped packets:",n+0}' out.tr

Dropped packets: 0
